# Input Tensor (`x`)

Given:

```python
B,T,C = 4,8,32
x = torch.randn(B,T,C)
```

---

## What do `B`, `T`, and `C` represent?

| Symbol | Meaning |
|----------|----------|
| B | Batch Size (number of sequences processed together) |
| T | Sequence Length (number of tokens) |
| C | Embedding Dimension (features per token) |

Example:

```python
(4,8,32)
```

means:

```text
4 sequences
8 tokens per sequence
32 features per token
```

---

## Why is `x.shape = (B,T,C)`?

Each token needs an embedding.

```text
Batch
 └── Tokens
      └── Features
```

Therefore:

```python
(B,T,C)
```

Example:

```text
Batch 0                                 Batch 1                    ....          Batch n
[                                  [
 token0 -> [32 numbers]               token8 -> [32 numbers]
 token1 -> [32 numbers]               token9 -> [32 numbers]
 ...                                  ...
 token7 -> [32 numbers]               token16 -> [32 numbers]
],                                  ],

```

---

## Why is `C` called embedding dimension?

Each token is represented by a vector:

```python
token = [x₁,x₂,x₃,...,x₃₂]
```

The number of values in that vector is the embedding dimension.

```python
C = len(token_embedding)
```

Example:

```python
[0.2, -1.1, 0.7]
```

↓

```python
C = 3
```

---

## Why is `T` sometimes called the time dimension?

Historically sequence models processed data step-by-step:

```text
t=0
t=1
t=2
...
```

For text:

```text
"I love transformers"
```

becomes:

```text
Token0 -> I
Token1 -> love
Token2 -> transformers
```

So:

```text
Time ≈ Position in sequence
```

Not necessarily actual clock time.

---

## What does `x[b,t]` contain?

Indexing removes one dimension:

```python
x.shape
=
(B,T,C)
```

```python
x[b,t]
```

Shape:

```python
(C,)
```

Contains:

```text
Embedding vector of token t
from batch b
```

Example:

```python
x[0,3]
```

↓

```python
[
 0.12,
 -0.44,
 1.23,
 ...
]
```

Length:

```python
C
```

---

## What does `x[b,t,c]` represent?

```python
x[b,t,c]
```

Shape:

```python
scalar
```

Meaning:

```text
Feature c
of token t
inside batch b
```

Example:

```python
x[0,3,5]
```

↓

```python
0.817
```

Interpretation:

```text
Batch 0
Token 3
Feature 5
```

---

## Shape Breakdown

| Expression | Shape | Meaning |
|------------|---------|---------|
| `x` | `(B,T,C)` | Entire tensor |
| `x[b]` | `(T,C)` | One sequence |
| `x[b,t]` | `(C,)` | One token embedding |
| `x[b,t,c]` | `()` | Single feature value |

---

## Mental Model

```text
x 
(B,T,C)
↓
Batch
↓
Tokens
↓
Embedding Features
```

Example:

```text
x[0]

[
 token0 -> [32 features]
 token1 -> [32 features]
 ...
]
```

Each token is represented by a vector.

Each vector has `C` learned features.

## `torch.randn`

Given:

```python
B,T,C = 4,8,32
x = torch.randn(B,T,C)
```

Shape:

```python
(4,8,32)
```

---

### What does `torch.randn(B,T,C)` generate?

Generates:

```python
B*T*C
```

independent random numbers sampled from a normal distribution.

Example:

```python
torch.randn(2,3)
```

↓

```python
[
 [ 0.12, -0.81, 1.34],
 [-1.02,  0.57, 0.21]
]
```

---

### How can a normal distribution exist in 3 dimensions?

It doesn't (here).

PyTorch is **not** generating a 3D Gaussian.

It simply generates:

```python
4 * 8 * 32
=
1024
```

independent random numbers and arranges them into:

```python
(4,8,32)
```

Think:

```text
Generate numbers first
↓
Arrange into tensor later
```

---

### Is it generating a 3D Gaussian?

❌ No

```python
torch.randn(4,8,32)
```

is equivalent to:

```python
Generate 1024 random values
↓
Reshape to (4,8,32)
```

Each element is sampled independently.

---

### Why random values?

Neural networks learn by adjusting weights.

If everything starts with:

```python
0
```

then every neuron behaves identically.

Example:

```text
Neuron 1 → same output
Neuron 2 → same output
Neuron 3 → same output
```

Gradients become identical.

Learning breaks.

---

### Why specifically a Normal Distribution?

`torch.randn()` samples from:

```text
N(0,1)

- **Mean = 0 (`μ`)** → Average of all generated values is 0 (centered at zero).
- **Std = 1 (`σ`)** → Controls spread; most values lie within `[-1,1]`(~68 %), almost all within `[-3,3]`(~99.7 %). (Extra [-2,2] (~95 %))
- Small random values → Stable initialization for training.
```
Typical values:

```text
≈ -3 to +3
```

Properties:

| Property | Benefit |
|----------|----------|
| Mean = 0 | No initial directional bias |
| Most values are small | Stable forward/backward pass |
| Positive & Negative values | Breaks symmetry between neurons |
| Few large values | Prevents huge activations initially |
| Random | Different neurons learn different features |
---

### Shape Breakdown

| Expression | Meaning |
|------------|---------|
| `torch.randn(32)` | 32 random values |
| `torch.randn(8,32)` | 8 vectors of length 32 |
| `torch.randn(4,8,32)` | 4 batches × 8 tokens × 32 features |

---

### Mental Model

```python
torch.randn(4,8,32)
```

↓

```text
Generate 1024 random numbers
↓
Arrange into

(B,T,C)

=
(4,8,32)
```

The tensor shape does **not** affect the distribution.

It only affects how the numbers are organized.

# Linear Layers (`nn.Linear`)

## What is `nn.Linear(in_features, out_features)`?

A learnable linear transformation:

```python
y = xWᵀ + b
```

Transforms:

```text
(*, in_features)
        ↓
(*, out_features)
```

Example:

```python
nn.Linear(32,16)
```

↓

```text
32 features → 16 features
```

---

# Is `nn.Linear` a Function, Tensor, Matrix, or Object?

## Comparison

| Type | What it stores | Can it be called? | Example |
|------|----------------|-------------------|---------|
| Function | Code | ✅ Yes | `print()`, `len()` |
| Tensor | Numbers | ❌ No | `torch.tensor([1,2,3])` |
| Object | Data + Functions | Sometimes (`__call__`) | `nn.Linear(...)` |

---

## Function

A function only contains instructions.

```python
def square(x):
    return x*x
```

Usage:

```python
square(5)

# 25
```

No internal state.

---

## Tensor

A tensor only stores numbers.

```python
x = torch.tensor([1,2,3])
```

Contains:

```text
[1,2,3]
```

It doesn't know how to transform another tensor.

---

## Object

An object stores:

```text
Data + Functions
```
Example:

```python
layer = nn.Linear(3,2) (where bias = True by default)
```

Inside `layer`:

```python
layer.weight
layer.bias
```

and also methods like:

```python
layer(x)
```

Internally, Python does:

```python
layer.__call__(x)
```

which eventually performs:

```python
x @ weight.T + bias
```

---

## Why is `nn.Linear` an object?

Because it needs to remember its parameters.

```python
layer = nn.Linear(3,2)

print(layer.weight)
print(layer.bias)
```

If it were just a function: linear(x)

where would the weights be stored?

They would have to be passed every time.

Instead: layer

owns its own:
      - weight matrix
      - bias vector
      - forward computation

Purpose:

```text
Break symmetry
Allow learning
```

---

## What does `bias=False` mean?

Without bias:

```python
y = xWᵀ
```

Only matrix multiplication.

```python
nn.Linear(32,16,bias=False)
```

---

## What changes when `bias=True`?

With bias:

```python
y = xWᵀ + b
```

Bias:

```python
b.shape = (out_features,)
```
---

## Shape Example

Input:

```python
x.shape = (B,T,32)
```

Layer:

```python
nn.Linear(32,16)  (in_feature, out_features)
```

Operation:

```text
(B,T,32) @ (32,16) = (B,T,16)
```
Only the last dimension changes.

---

## Why only the last dimension?

Documentation:

```text
(*, Hin)
```

means:

```text
Anything before Hin is preserved
Only Hin is transformed
```

Examples:

| Input | Output |
|---------|---------|
| `(32)` | `(16)` |
| `(8,32)` | `(8,16)` |
| `(4,8,32)` | `(4,8,16)` |
| `(100,4,8,32)` | `(100,4,8,16)` |

---

---

## Quick Reference

| Question | Answer |
|----------|---------|
| What is it? | Learnable linear transform |
| Stores parameters? | Yes |
| Parameters? | Weight + Bias |
| Weight shape | `(out_features, in_features)` |
| Bias shape | `(out_features,)` |
| Initialized by | PyTorch |
| `bias=False` | No bias term |
| `bias=True` | Adds bias |
| Changes which dimension? | Last dimension only |

### Core Formula

```python
nn.Linear(Hin, Hout)
```

```text
(*, Hin)
    ↓
(*, Hout)
```

```text
(B,T,Hin) @ (Hin,Hout) = (B,T,Hout)
```

## Why can `nn.Linear` accept 3D tensors?

Because it operates independently on every vector whose length is:

```python
Hin
```

Example:

```python
x.shape = (4,8,32)
```

Think:

```text
32-d vector
32-d vector
32-d vector
...
```

There are:

```python
4 * 8 = 32
```

such vectors.

Each one gets transformed:

```text
(1×32) @ (32×16) = (1×16)
```
Result: (4,8,16)
```

## Visual Example

```python
x.shape = (2,3,4)
```

```text
Batch 0                 Batch 1
[                       [
 token0 -> [4 nums]      token0 -> [4 nums]
 token1 -> [4 nums]      token1 -> [4 nums] 
 token2 -> [4 nums]      token2 -> [4 nums]  
]                       ]

```

Apply:

```python
nn.Linear(4,2)
```

Each token:

```text
(1×4) @ (4×2) = (1×2)
```
Result: (2,3,2)

# Linear Layers: Mathematics & Learning

## What operation is actually performed inside `nn.Linear`?

Without bias:

```python
y = xWᵀ
```

With bias:

```python
y = xWᵀ + b
```

Core operation:

```text
Matrix Multiplication
```

---

## Is it just matrix multiplication?

✅ Yes

```python
nn.Linear(Hin,Hout)
```

is essentially:

```text
(*,Hin) @ (Hin×Hout) = (*,Hout)
```

plus optional bias.

---

## How does a linear layer combine features?

Example:

```python
x = [1,2,3]
```

```python
W =
[
 [1,0],
 [0,1],
 [1,1]
]
```

```text
(1×3) @ (3×2) = (1×2)
```

Output:

```python
[
 1*1 + 2*0 + 3*1, 1*0 + 2*1 + 3*1
]
```

↓

```python
[4,5]
```

Each output is a weighted combination of all inputs.

---

## Why is this called a projection?

Input:

```python
[1,2,3]
```

Output:

```python
[4,5]
```

Same information source.

Different coordinate system.

```text
Embedding
     ↓
Linear Transform
     ↓
Projected Embedding
```

The vector is being represented in a new space.

---

## Why not simply call it a new embedding?

Because:

```text
New embedding
```

doesn't explain where it came from.

```text
Projection
```

implies:

```text
Original vector
      ↓
Linear transformation
      ↓
New representation
```

The relationship is preserved.

---

# Learning

## How does a linear layer learn which features matter?

Initially:

```python
W = random
```

Nothing meaningful exists.

During training:

```text
Forward Pass
     ↓
Loss
     ↓
Backpropagation
     ↓
Update Weights
```

Repeated millions of times.

---

## If weights start random, how do meaningful features emerge?

Initially:

```text
Feature 7 weight = random
Feature 12 weight = random
Feature 25 weight = random
```

After training:

```text
Useful features
↑ larger weights

Less useful features
↑ smaller weights
```

The optimizer discovers which combinations reduce loss.

---

## Is this entirely due to backpropagation?

Almost.

Learning pipeline:

```text
Random Initialization
        ↓
Forward Pass
        ↓
Loss
        ↓
Backpropagation
        ↓
Gradient Descent
        ↓
Updated Weights
```

Backprop computes:

```text
How should weights change?
```

Gradient descent applies the change.

---

## What if `out_features < in_features`?

Example:

```python
nn.Linear(32,16)
```

```text
32 → 16
```

Compression.

```text
Keep important information
Discard redundant information
```

Shape:

```text
(B,T,32)
@
(32,16)
=
(B,T,16)
```

---

## What if `out_features = in_features`?

Example:

```python
nn.Linear(32,32)
```

```text
32 → 32
```

No compression.

No expansion.

Still useful because:

```text
Old Features
      ↓
Mixed Together
      ↓
New Features
```

Shape:

```text
(B,T,32)
@
(32,32)
=
(B,T,32)
```

Representation changes.

Dimension stays the same.

---

## What if `out_features > in_features`?

Example:

```python
nn.Linear(32,64)
```

```text
32 → 64
```

Expansion.

Creates more feature channels.

Shape:

```text
(B,T,32)
@
(32,64)
=
(B,T,64)
```

Common in Transformer MLPs:

```text
d_model
   ↓
4*d_model
   ↓
d_model
```

## Quick Reference

| Case | Meaning |
|--------|--------|
| `32 → 16` | Compression |
| `32 → 32` | Re-mapping |
| `32 → 64` | Expansion |
| Random weights | Initialization |
| Backprop | Computes updates |
| Gradient Descent | Applies updates |
| Projection | Same information, new space |
| Linear Layer | Learn feature combinations |

# Design Questions

## Why use `Linear` instead of `BiLinear`?

Goal of Q/K projection:

```text
Embedding
      ↓
Learn a new representation space
```

Linear already does exactly that:

```python
q = xWq
k = xWk
```

BiLinear needs two inputs:

```python
y = x₁ᵀAx₂
```

Attention hasn't started comparing tokens yet.

We first need to transform each token independently.

---

## Why is a linear transformation sufficient here?

A linear layer can:

```text
Rotate
Scale
Stretch
Compress
Mix Features
```

which is enough to create a new representation space.

Example:

```text
32 features
     ↓
Linear
     ↓
16 learned features
```

The actual non-linearity comes later from:

```python
softmax()
```

and other layers in the network.

---

## What properties of linear transformations make them useful?

| Property | Benefit |
|-----------|-----------|
| Learnable | Can adapt during training |
| Efficient | Fast matrix multiplication |
| Differentiable | Easy to optimize |
| Mixes features | Creates richer representations |
| Dimension change | Compress or expand features |

---

# Keys and Queries

## Why are `key` and `query` different if both are:

```python
nn.Linear(32,16)
```

Because:

```python
key   = nn.Linear(32,16)
query = nn.Linear(32,16)
```

creates separately.

Each layer gets its own random weights.




# Conceptual

## How does one projection become a "query"?

Initially:

```text
Query = random projection
```

Nothing in the code says:

```text
"This is searching."
```

During training:

```text
Predict next token
      ↓
Backpropagation
      ↓
Update Wq
```

The model discovers:

```text
Some features are useful for asking:
"What information do I need?"

Those become query features.
Same for key features.

```


## Quick Reference

| Question | Answer |
|-----------|-----------|
| Why Linear? | Learn a new representation space |
| Why different Q/K? | Different weight matrices |

# Representation Learning

## What does "projection of an embedding" actually mean?

Original embedding:

```python
x = [1,2,3]
```

Apply:

```python
q = xWq
```

```text
(1×3) @ (3×2) = (1×2)
```
Result: q = [4,5]

Same token.

Different coordinates.

```text
Embedding
     ↓
Linear Transform
     ↓
Projected Embedding
```

---

## What is a representation space?

A space where vectors live.

Example:

```python
[1,2,3]
```

lives in:

```text
R³
```

After:

```python
xWq
```

↓

```python
[4,5]
```

it lives in:

```text
R²
```

Different space.

Different coordinates.

Different notion of similarity.

---

## Why learn a new representation space?

Raw embeddings contain:

```text
Grammar
Syntax
Meaning
Position
Etc.
```

Not all information is useful for attention.

Learned projections can focus on:

```text
Features useful for matching
```

Example:

```text
Raw Embedding
      ↓
Projection
      ↓
Only useful matching signals remain
```

---

## Why not work directly in embedding space?

Possible:

```python
x @ x.T
```

```text
(T×C) @ (C×T) = (T×T)
```

But then:

```text
Similarity is forced to use
the raw embedding space.
```

No learning.

No specialization.

With:

```python
q = xWq
k = xWk
```

the model first learns:

```text
What features should matter?
```

and then computes similarity.

---

## Mental Model

```text
Embedding Space
       ↓
Learn Projection
       ↓
Query/Key Space
       ↓
Measure Similarity
```

---

# Matrix Multiplication

## How does batch matrix multiplication work?

PyTorch treats:

```python
(B,T,16)
```

as:

```text
B separate matrices
```

Example:

```python
(4,8,16)
```

means:

```text
Matrix 0 -> (8×16)
Matrix 1 -> (8×16)
Matrix 2 -> (8×16)
Matrix 3 -> (8×16)
```

and

```python
(4,16,8)
```

means:

```text
Matrix 0 -> (16×8)
Matrix 1 -> (16×8)
Matrix 2 -> (16×8)
Matrix 3 -> (16×8)
```

PyTorch computes:

```text
(8×16) @ (16×8) = (8×8)
```

for each batch independently.

Result:

```python
(4,8,8)
```

## Quick Reference

| Expression | Result |
|------------|---------|
| `(m×n) @ (n×o)` | `(m×o)` |
| `(T×16) @ (16×T)` | `(T×T)` |
| `(T×16) @ (T×16)` | ❌ Invalid |
| `x @ x.T` | Similarity in embedding space |
| `q @ k.T` | Similarity in learned space |
| `(B,T,16) @ (B,16,T)` | `(B,T,T)` |

### Mental Model

```text
q = What am I looking for?
k = What information do I contain?

q @ k.T

=
How relevant is token j
to token i?
```

# Dot Product

## What is the dot product actually computing?

For two vectors:

```python
a = [a₁,a₂,...]
b = [b₁,b₂,...]
```

Dot product:

$$
a \cdot b
=
\sum_i a_i b_i
$$

Example:

```python
[1,2,3]
·
[4,5,6]
```

```text
=
1×4 + 2×5 + 3×6

=
32
```

---

## Why does it measure similarity?

Each feature votes:

```text
Same sign  → positive contribution
Opposite sign → negative contribution
Near zero → little contribution
```

Example:

| Vectors | Dot Product | Interpretation |
|----------|----------|----------|
| `[1,2] · [1,2]` | `5` | Similar |
| `[1,2] · [-1,-2]` | `-5` | Opposite |
| `[1,0] · [0,1]` | `0` | Unrelated |

---

## What does a large dot product mean?

Large positive:

```text
Features agree
```

Example:

```python
[3,4]
·
[2,5]
=
26
```

Many dimensions point in the same direction.

---

## What does a negative dot product mean?

Example:

```python
[1,2]
·
[-1,-2]
=
-5
```

Features disagree.

The vectors point in opposite directions.



# Attention

## What exactly is:

```python
q @ k.T
```

measuring?

Shapes:

```text
(T×16)
@
(16×T)
=
(T×T)
```

Each cell:

```text
q(token_i)
·
k(token_j)
```

Example:

```text
[
 q₀·k₀  q₀·k₁  q₀·k₂
 q₁·k₀  q₁·k₁  q₁·k₂
 q₂·k₀  q₂·k₁  q₂·k₂
]
```

Every token compared with every token.

---

## Is it important?

❌ Not directly.

A token can be important but irrelevant to the current token.

Dot product does NOT ask:

```text
"How important are you?"
```

---

## Is it similarity?

✅ Yes.

It measures:

```text
How similar is my query
to your key?
```

---

## Is it relevance?

✅ More accurate than "importance".

The attention score asks:

```text
How useful are you
for what I'm currently looking for?
```

---

## Similarity vs Importance

| Concept | Meaning |
|----------|----------|
| Importance | Important overall |
| Similarity | Matches my features |
| Relevance | Useful for my current query |

Attention mainly learns:

```text
Relevance
```


---

## What kinds of features are being compared?

The model decides.

Examples:

```text
Subject information
Location information
Gender information
Verb agreement
Entity references
```

No human explicitly defines them.

Training discovers useful features.

---

## What is the dot product measuring after projection?

Before projection:

```python
x @ x.T
```

measures:

```text
Similarity in embedding space
```

After projection:

```python
q = xWq
k = xWk
```

```python
q @ k.T
```

measures:

```text
Similarity in a learned space
```

where:

```text
Wq and Wk learned
which features should matter.
```

---

## Full Pipeline

```text
Embedding
      ↓
q = xWq
k = xWk
      ↓
Learn useful features
      ↓
q @ k.T
      ↓
Similarity Scores
      ↓
Softmax
      ↓
Attention Weights
```

---

## The Deepest Interpretation

```python
q @ k.T
```

does NOT ask:

```text
Who is important?
```

It asks:

```text
Given what token i is looking for,
which token j best matches it?
```

That matching score is the dot product.

---

## Quick Reference

| Expression | Meaning |
|------------|----------|
| `a · b` | Feature agreement |
| Large positive | Strong match |
| Negative | Opposite features |
| `x @ x.T` | Similarity in embedding space |
| `q @ k.T` | Similarity in learned space |
| Attention score | Relevance score |
| Softmax output | Attention weight |

### Mental Model

```text
Query:
"What am I looking for?"

Key:
"What information do I contain?"

Dot Product:
"How well do we match?"
```

# Matrix Multiplication Shape Rules

## Why is the batch dimension untouched?

Given:

```python
(B,T,16)
@
(B,16,T)
```

PyTorch performs:

```text
Batch 0:
(T×16) @ (16×T)

Batch 1:
(T×16) @ (16×T)

Batch 2:
(T×16) @ (16×T)
```

independently.

The batch dimension is simply:

```text
How many matrices do we have?
```

not part of the matrix itself.

---

# Alternatives

## Why use dot product instead of cosine similarity?

Cosine similarity:

$$
\cos(\theta)
=
\frac{a \cdot b}{||a||\,||b||}
$$

Measures:

```text
Direction only
```

Ignores magnitude.

---

### Example

```python
a = [1,1]
b = [10,10]
```

Cosine:

```text
1.0
```

because they point in the same direction.

Dot Product:

```text
20
```

because both direction and magnitude matter.

---

## Why does attention prefer dot products?

| Reason | Benefit |
|----------|----------|
| Faster | No norm computation |
| Simple | Just matrix multiplication |
| Differentiable | Easy optimization |
| Magnitude matters | Strong signals can be emphasized |
| GPU friendly | Extremely efficient |

---

## Then why scale by √d?

Large dimensions:

```python
q·k
```

can become very large.

Large values:

```python
softmax(...)
```

↓

```text
Very peaky probabilities
```

Small gradients.

Therefore:

```python
(q @ k.T) / √d
```

keeps values stable.

---

# Attention Scores (`wei`)

## Why is:

```python
wei.shape = (B,T,T)
```

Given:

```python
q.shape = (B,T,16)
k.shape = (B,T,16)
```

Transpose:

```python
k.T
```

↓

```python
(B,16,T)
```

Multiply:

```text
(B,T,16) @ (B,16,T) = (B,T,T)
```
because:

```text
(T×16) @ (16×T) = (T×T)
```

---

## Why do we get a token-token matrix?

Every token compares itself against every token.

Example:

```text
Token0 vs Token0
Token0 vs Token1
Token0 vs Token2

Token1 vs Token0
Token1 vs Token1
Token1 vs Token2

...
```

Number of comparisons:

```text
T × T
```

Therefore:

```python
(T,T)
```

---

## What does each cell contain?

Cell:

```python
wei[i,j]
```

contains:

```text
q(token_i) · k(token_j)
```

which is:

```text
Similarity / Relevance Score
```

before softmax.

## What does:

```python
wei[i,j]
```

mean?

It answers:

```text
How relevant is token j
to token i?
```

or

```text
How much should token i
pay attention to token j?
```

Notice the direction:

```text
Row = Querying token
Column = Candidate token
```

---

### Example

Sentence:

```text
"The cat sat"
```

Suppose:

```python
wei[2,1]
```

Then:

```text
Row 2 → "sat"
Column 1 → "cat"
```

Meaning:

```text
How relevant is "cat"
to "sat"?
```

before softmax normalization.

---

## Before vs After Softmax

Before:

```python
wei
```

```text
Raw similarity scores
```

Example:

```python
[
 [2,5,1]
]
```

After:

```python
softmax(wei)
```

```python
[
 [0.05,0.91,0.04]
]
```

Now:

```text
Attention weights

that sum to: 1
```

---

## Quick Reference

| Expression | Meaning |
|------------|----------|
| `x @ x.T` | Similarity in embedding space |
| `q @ k.T` | Similarity in learned space |
| `wei.shape` | `(B,T,T)` |
| Row `i` | Query token |
| Column `j` | Candidate token |
| `wei[i,j]` | Relevance of token `j` to token `i` |
| Before Softmax | Scores |
| After Softmax | Attention Weights |

### Mental Model

```text
Query:
"What am I looking for?"

Key:
"What information do I contain?"

wei[i,j]:
"How well does token j
match what token i needs?"
```

# Raw Embeddings vs Q/K

## Why not simply do:

```python
x @ x.T
```

Shapes:

```text
(T×C) @ (C×T) = (T×T)
```

This would measure:

```text
Similarity in the raw embedding space
```

The problem:

```text
The model cannot choose
which features matter.
```

It is forced to use whatever information happens to be stored in the embedding.

---

## Why introduce `Wq` and `Wk`?

Instead:

```python
q = xWq
k = xWk
```

Now attention becomes:

```python
q @ k.T
```

```text
(T×16) @ (16×T) = (T×T)
```

The model first learns:

```text
Which features should be compared?
```

and only then computes similarity.

---

## Raw Embeddings vs Learned Projections

| Raw Embeddings | Q/K Projections |
|---------------|---------------|
| Fixed similarity | Learned similarity |
| Uses all features equally | Learns useful features |
| No specialization | Query/Key specialization |
| `x @ x.T` | `q @ k.T` |
| Generic matching | Task-specific matching |

---

# Learning

## What additional flexibility do learned projections provide?

Without projections:

```text
Token
    ↓
Embedding
    ↓
Similarity
```

With projections:

```text
Token
    ↓
Embedding
    ↓
Learn Features
    ↓
Similarity
```

The extra step allows the model to decide:

```text
What should matter?
```

instead of us deciding.

---

## How do projections amplify useful features?

Suppose embedding:

```python
[
 subject,
 tense,
 color,
 position
]
```

Maybe attention only needs:

```text
subject
position
```

Training may learn:

```text
Large weights
for useful features

Small weights
for irrelevant features
```

Example:

```python
[2,3,1,4]
```

↓

```python
[10,12]
```

Useful information becomes stronger.

---

## How does this happen mathematically?

Suppose:

```python
x = [2,5,1]
```

and:

```python
W =
[
 [5],
 [4],
 [0.1]
]
```

Then:

```text
(1×3) @ (3×1) = (1×1)
```

```python
2×5 + 5×4 + 1×0.1 = 30.1
```

Feature 1 and 2 dominate.

Feature 3 barely contributes.

The projection has amplified important signals.

---

## Is attention a kind of feature amplification?

✅ Partly yes.

Q and K layers learn:

```text
Which features deserve attention?
```

The projection can strengthen:

```text
important features
```

and weaken:

```text
unimportant features
```

before similarity is computed.

---

## Is attention a kind of feature extraction?

✅ Also yes.

This is the subtle part.

The projection creates:

```text
New Features
```

not merely stronger old ones.

Example:

```text
Old Features:
gender
tense
position
```

↓

```text
New Learned Feature:
"is this token likely
the subject of a sentence?"
```

That feature did not explicitly exist before.

---

## Amplification vs Extraction

| Amplification | Extraction |
|---------------|-------------|
| Increase useful signals | Create new signals |
| Larger weights | New feature combinations |
| Keep existing information | Transform information |
| "Make feature stronger" | "Invent better feature" |

Attention does both.


# Computing Attention Scores (`wei`)

```python
q = xWq
k = xWk
v = xWv   # (introduced in the next version)
```

All three are **learned projections** of the same embedding.

| Projection | Learns |
|------------|--------|
| **Query (Q)** | What am I looking for? |
| **Key (K)** | What information do I contain? |
| **Value (V)** | What information should I send if attended to? |

Shapes:

```text
q : (B,T,16)
k : (B,T,16)
v : (B,T,16)    # later version
```

---

## What does one batch look like?

For one batch (`B=0`):

```text
q[0]

          Features →
        f0  f1  f2 ... f15
Token0   •   •   •  ...  •
Token1   •   •   •  ...  •
Token2   •   •   •  ...  •
...
Token7   •   •   •  ...  •

Shape = (T×16)
```

Each **row** is one token.

Each **column** is one learned feature.

---

## Why transpose?

```python
k.transpose(-2,-1)
```

Before:

```text
(T×16)
```

After:

```text
(16×T)
```

```text
k[0].T

          Tokens →
        T0  T1  T2 ... T7
feature0       •   •   •  ...  •
feature1       •   •   •  ...  •
feature2       •   •   •  ...  •
...
feature15      •   •   •  ...  •

Shape = (16×T)
```

Notice:

- Each **column** = one token.
- Each **row** = values of one learned feature across all tokens.

Transpose simply rotates the matrix.

---

## Matrix Multiplication

```python
wei = q @ k.transpose(-2,-1)
```

```text
(B,T,16) @ (B,16,T) = (B,T,T)
```

---

## Where did the 16 features go?

Each output cell is:

```text
q(token_i) · k(token_j)
```

Expanded:

```text
(f0, f1, ..., f15) · (f0, f1, ..., f15)
```
                   ↓

```text
f0×f0 + f1×f1 + ... + f15×f15
```

The **16 feature comparisons are reduced to one scalar**.

```text
16 Features
      ↓
Dot Product
      ↓
1 Similarity Score
```

That's why:

```text
(T×16) @ (16×T) = (T×T)
```

---

## What does `(T,T)` contain?

```text
[
 q₀·k₀  q₀·k₁  q₀·k₂ ...
 q₁·k₀  q₁·k₁  q₁·k₂ ...
 q₂·k₀  q₂·k₁  q₂·k₂ ...
 ...
]
```

Each cell answers:

```text
"How relevant is token j
to token i?"
```

These are **raw attention scores**.

---

## After Softmax

```python
wei = F.softmax(wei, dim=-1)
```

Each row becomes:

```text
Attention Weights
```

Example:

```text
[2,5,1]
```

↓

```text
[0.05,0.91,0.04]
```

Meaning:

```text
Token attends

5%  → Token0
91% → Token1
4%  → Token2
```

Each row now sums to:

```text
1
```

---

## Why multiply by `x`?

In Andrej's notebook:

```python
out = wei @ x
```

```text
(B,T,T) @ (B,T,C) = (B,T,C)
```

Ignoring the batch:

```text
(T×T) @ (T×C) = (T×C)
```

The attention weights decide:

```text
How much information
to take from each token.
```

Example:

```text
Weights

[0.7,0.2,0.1]
```

↓

```text
0.7×Token0 + 0.2×Token1 + 0.1×Token2
```

Result:

```text
Weighted Average
of token embeddings
```

---

## Where does Value (V) come in?

In the full Transformer:

```python
v = xWv

out = wei @ v
```

instead of:

```python
out = wei @ x
```

Reason:

| Q & K | V |
|--------|---|
| Decide **where** to look | Contains **what** information to retrieve |

The complete attention equation is:

```text
Attention(Q,K,V)

=

softmax(QKᵀ)V [(B,T,T) @ (B,T,C)]
```

---

## Complete Flow

```text
Embedding (x)
      │
      ├──► Query (Q)
      ├──► Key (K)
      └──► Value (V)

Q @ Kᵀ
      ↓
Similarity Scores
      ↓
Softmax
      ↓
Attention Weights
      ↓
Weighted Average of V
      ↓
Context-Aware Token Representations
```

---

## Mental Model

```text
Q:
"What am I looking for?"

K:
"What information do I contain?"

V:
"What information should I send?"

Attention:
"Find the right tokens,
then gather their information."
```

# Scaled Dot-Product Attention

The complete attention equation is:

```python
Attention(Q,K,V) = softmax((QKᵀ)/√d)V
```

where:

```text
d = head_size
```

---

## Why scale at all?

Each attention score is:

```text
q · k
```

Expanded:

```text
q₁k₁ + q₂k₂ + ... + q_dk_d
```

As `d` (head size) increases:

- More multiplications
- More additions
- Larger dot products

Large scores make softmax become:

```text
[0.0001, 0.9998, 0.0001]
```

which is extremely "peaky".

This causes:

- Tiny gradients
- Slow/unstable training

---

## Why specifically √d?

Suppose every feature has:

```text
Mean = 0
Variance = 1
```

(one reason weights are initialized this way)

Each product:

```text
qᵢkᵢ
```

has roughly:

```text
Variance ≈ 1
```

Now add **d** independent terms:

```text
q₁k₁ + ... + q_dk_d
```

Variance becomes:

```text
Var(sum) = d
```

Standard deviation is:

```text
Std = √Var = √d
```

Therefore:

```text
Typical dot product size ≈ √d
```

---

## Scaling

Divide by:

```text
√d
```

Now:

```text
Typical magnitude

≈ 1
```

regardless of head size.

So whether:

```text
d = 16
```

or

```text
d = 1024
```

the attention scores stay on a similar scale.

---

## Why NOT divide by `d`?

Suppose:

```text
d = 64
```

Typical dot product:

```text
≈ √64 = 8
```

Correct scaling:

```text
8 / 8 = 1
```

---

If we divide by:

```text
64
```

instead:

```text
8 / 64 = 0.125
```

Everything becomes much too small.

Softmax then produces:

```text
Almost Uniform

[0.33,0.34,0.33]
```

Attention becomes too "flat".

The model struggles to focus.

---

## Comparison

| Divide By | Effect |
|-----------|--------|
| Nothing | Scores become too large |
| √d | Stable attention scores ✅ |
| d | Scores become too small |

---

## Intuition

Suppose you flip coins.

More flips:

```text
More total variation
```

But variation grows like:

```text
√N
```

not:

```text
N
```

The same statistical idea applies to the dot product.

---

## Mental Model

```text
Dot Product

↓

Magnitude grows as √d

↓

Divide by √d

↓

Scores stay around 1

↓

Healthy Softmax

↓

Stable Gradients
```

---

## Quick Reference

| Quantity | Grows Like |
|----------|------------|
| Number of summed terms | `d` |
| Variance of dot product | `d` |
| Standard deviation | `√d` |
| Scaling factor | `1/√d` |

### One-Line Summary

```text
The dot product sums d independent terms.

Its variance grows as d, so its standard deviation grows as √d.

Dividing by √d keeps attention scores at a consistent scale, preventing softmax from becoming too sharp or too flat.
```

#### Variance = Average squared distance from the mean.

Bigger head
↓

More feature comparisons

↓

Larger dot products
{softmax([-40,15,32,55]) ≈ [0,0,0,1]
The attention becomes almost one-hot.}

↓

Softmax becomes too confident

↓

Divide by √head_size (softmax saturation, which then leads to vanishing gradients)

↓

Bring scores back to the same scale

↓

Softmax behaves consistently

```text
You're trying to normalize the typical size (spread) of the values.

The spread is measured by the standard deviation, not the variance.
```